# Объявления с метками из опросов

Берём готовые метки из `survey_deal_labels.csv` и добавляем все поля из исходной таблицы объявлений по `listing_id`. Запускать из корня репозитория.

## 1. Загрузить объявления и метки

In [4]:
from pathlib import Path

import pandas as pd
from IPython.display import display

LISTINGS_PATH = Path("data/listings.csv")
LABELS_PATH = Path("data_by_dewallen/survey_deal_labels.csv")
OUTPUT_PATH = Path("data_by_dewallen/survey_deal_listings.csv")

listings = pd.read_csv(LISTINGS_PATH)
labels = pd.read_csv(LABELS_PATH)

print("Строк в таблице объявлений:", len(listings))
print("Строк с метками:", len(labels))

Строк в таблице объявлений: 15457
Строк с метками: 2480


## 2. Присоединить метки по listing_id

Убираем точные дубликаты из объявлений, чтобы каждый ID из таблицы меток дал одну строку.

In [5]:
unique_listings = listings.drop_duplicates()
assert labels["listing_id"].notna().all(), "В метках есть пустые ID"
assert labels["listing_id"].isin(unique_listings["listing_id"]).all(), "Для некоторых меток нет объявления"

labeled_listings = unique_listings.merge(
    labels, on="listing_id", how="inner", validate="one_to_one",
)
assert len(labeled_listings) == len(labels), "После объединения изменилось число объявлений"
assert labeled_listings.columns.tolist() == [*listings.columns, "is_deal", "weight"]


labeled_listings = labeled_listings.sort_values("phone_views_total")
labeled_listings = labeled_listings.sort_values("views_total")
labeled_listings = labeled_listings.drop(index=labeled_listings[(labeled_listings["views_total"] == 0) & (labeled_listings["phone_views_total"] == 0)].index)
print("Объявлений с метками:", len(labeled_listings))

Объявлений с метками: 2437


## 3. Сохранить итоговую таблицу

In [6]:
display(labeled_listings.shape)
labeled_listings = labeled_listings.sort_values("listing_id")
labeled_listings.to_csv(OUTPUT_PATH, index=False)

print("Сохранено в", OUTPUT_PATH)

(2437, 20)

Сохранено в data_by_dewallen/survey_deal_listings.csv
